# 01. Análisis exploratorio de clientes

**Desarrollado por:** Alan Arturo Hernández Gutiérrez

**Objetivo:** revisar el dataset, entender qué perfiles presentan más bajas y decidir qué datos conviene preparar para Power BI.

El análisis parte del CSV original. No contiene fechas de abandono ni resultados reales de campañas.


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
BASE = Path.cwd().resolve()
if BASE.name.lower() == "notebooks":
    BASE = BASE.parent
assert (BASE / "input").is_dir(), "Inicia Jupyter desde la raiz del proyecto o desde notebooks/."
RUTA_INSUMO = BASE / "input" / "Bank Customer Churn Prediction.csv"
assert RUTA_INSUMO.is_file(), f"No existe el CSV original: {RUTA_INSUMO}"
df = pd.read_csv(RUTA_INSUMO, encoding="utf-8-sig")
print("Archivo:", RUTA_INSUMO)
print("Dimensiones:", df.shape)
display(df.head(5))

## Lectura de insumos

Se usa el archivo original de clientes. Primero reviso su estructura y después comparo las variables con el abandono. Las transformaciones finales se hacen en el segundo notebook.


In [ ]:
columnas = ["customer_id","credit_score","country","gender","age","tenure","balance","products_number","credit_card","active_member","estimated_salary","churn"]
assert df.columns.tolist() == columnas
assert len(df) == 10000 and df.customer_id.is_unique
assert int(df.churn.sum()) == 2037
resumen = pd.DataFrame({"observacion": ["Clientes", "Bajas", "No bajas", "Churn observado", "Saldo agregado de promedios"], "valor": [len(df), int(df.churn.sum()), int((df.churn==0).sum()), df.churn.mean(), df.balance.sum()]})
display(resumen)

## 1. Conocer las columnas

Cada fila es un cliente. `customer_id` identifica al cliente y `churn` indica si abandonó (1) o no (0). No hay una fecha de abandono ni una causa registrada.

| Campo | Para qué sirve |
|---|---|
| `customer_id` | Evitar clientes repetidos |
| `credit_score` | Revisar diferencias por score |
| `country`, `gender` | Comparar países y género |
| `age`, `tenure` | Revisar edad y antigüedad |
| `balance`, `estimated_salary` | Revisar saldo y salario estimado |
| `products_number`, `credit_card`, `active_member` | Revisar productos, tarjeta y actividad |
| `churn` | Calcular clientes que abandonaron y tasa de abandono |

Los campos originales se mantienen en el ETL. El saldo y el salario no tienen moneda indicada.


### 1.1. Calidad de datos

Reviso nulos, clientes duplicados, tipos de datos y valores que conviene revisar. Una alerta no significa automáticamente que el dato esté mal.


In [ ]:
# Revisar los datos sin modificarlos.
chequeo_campos = pd.DataFrame({
    "campo": df.columns,
    "tipo": [str(df[c].dtype) for c in df.columns],
    "nulos": [int(df[c].isna().sum()) for c in df.columns],
    "valores_distintos": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
display(chequeo_campos)
calidad_origen = pd.DataFrame([
    ("Filas", len(df)),
    ("IDs duplicados", int(df.customer_id.duplicated().sum())),
    ("Nulos totales", int(df.isna().sum().sum())),
    ("Churn fuera de 0/1", int((~df.churn.isin([0, 1])).sum())),
    ("Tenure igual a 0 — revisión", int(df.tenure.eq(0).sum())),
    ("Score en 350 u 850 — revisión", int(df.credit_score.isin([350, 850]).sum())),
    ("Salario estimado menor a 100 — revisión", int(df.estimated_salary.lt(100).sum())),
    ("Edad menos tenure menor a 18 — revisión", int((df.age - df.tenure).lt(18).sum()))
], columns=["control", "resultado"])
display(calidad_origen)
assert int(df.customer_id.duplicated().sum()) == 0
assert int(df.isna().sum().sum()) == 0
assert set(df.churn.unique()) == {0, 1}
assert calidad_origen.iloc[4:]["resultado"].tolist() == [413, 238, 4, 322]

## 2. Primer análisis de abandono

Comparo países, actividad, productos, género y tarjeta. Siempre reviso **cantidad de clientes y tasa de abandono**: un porcentaje alto sobre pocos clientes puede confundir.


In [ ]:
def perfil(cols):
    g = df.groupby(cols,dropna=False).agg(clientes=("customer_id","size"), bajas=("churn","sum"), saldo_agregado=("balance","sum")).reset_index()
    g["churn_pct"] = 100*g.bajas/g.clientes
    g["indice_vs_global"] = (g.bajas/g.clientes)/df.churn.mean()
    return g.sort_values("bajas",ascending=False)
for campo in ["country","active_member","products_number","gender","credit_card"]:
    print("\n",campo)
    display(perfil([campo]))

### 2.1. Comparación visual

Estos gráficos sirven para ver diferencias iniciales. En Power BI se podrán explorar después con filtros.


In [ ]:
import matplotlib.pyplot as plt

def grafico_tasas_exploratorias(tabla, etiquetas, titulo):
    """Gráfico descriptivo temporal; no genera archivos ni modifica el dataframe fuente."""
    fig, ax = plt.subplots(figsize=(9.2, 3.7))
    tasas = tabla["churn_pct"].to_numpy()
    ax.bar(range(len(tabla)), tasas, width=.58)
    ax.axhline(100 * df.churn.mean(), linestyle="--", linewidth=1, label="Promedio total: 20.37%")
    ax.set_xticks(range(len(tabla)))
    ax.set_xticklabels(etiquetas)
    ax.set_ylabel("Abandono observado (%)")
    ax.set_title(titulo, loc="left", weight="bold")
    ax.set_ylim(0, max(float(max(tasas)) * 1.20, 26))
    for j, (tasa, n) in enumerate(zip(tasas, tabla["clientes"])):
        ax.text(j, tasa + max(tasas)*.027, f"{tasa:.1f}%  ·  n={n:,}", ha="center", fontsize=9)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    plt.show()

paises_eda = perfil(["country"]).sort_values("country").reset_index(drop=True)
grafico_tasas_exploratorias(paises_eda, paises_eda.country.map({"France":"Francia","Germany":"Alemania","Spain":"España"}).tolist(), "País — diferencias descriptivas")

In [ ]:
actividad_eda = perfil(["active_member"]).sort_values("active_member").reset_index(drop=True)
grafico_tasas_exploratorias(actividad_eda, ["Inactivo", "Activo"], "Actividad — perfil observado")
productos_eda = perfil(["products_number"]).sort_values("products_number").reset_index(drop=True)
grafico_tasas_exploratorias(productos_eda, [str(x) for x in productos_eda.products_number], "Número de productos — tasa y tamaño de muestra")

**Lo que se observa:** Alemania muestra más abandono que Francia y España. También destacan los clientes inactivos y los de tres o cuatro productos. El grupo de cuatro productos es pequeño (`n=60`), así que conviene revisarlo antes de proponer una acción. Son asociaciones, no causas demostradas.


### 2.2. Score, antigüedad y salario

Reviso también las variables numéricas. Para score y salario uso quintiles sólo para comparar; para antigüedad conservo los años. Las agrupaciones de esta parte no se guardarán como columnas adicionales.


In [ ]:
def resumen_grupos_exploratorio(variable, serie_agrupadora):
    tmp = df.assign(grupo_eda=serie_agrupadora)
    out = tmp.groupby("grupo_eda", observed=True, dropna=False).agg(
        clientes=("customer_id", "size"), bajas=("churn", "sum")
    ).reset_index()
    out["churn_pct"] = (100 * out.bajas / out.clientes).round(2)
    out.insert(0, "variable", variable)
    return out

for variable in ["credit_score", "tenure", "estimated_salary"]:
    if variable == "tenure":
        grupos_eda = df[variable]
    else:
        # Agrupar para comparar, sin cambiar el archivo.
        grupos_eda = pd.qcut(df[variable], q=5, duplicates="drop")
    tabla_eda = resumen_grupos_exploratorio(variable, grupos_eda)
    print("\n", variable, "— distribución y churn por grupo")
    display(tabla_eda)

print("Resumen descriptivo de las tres variables numéricas (sin convertirlas en predictores causales):")
display(df[["credit_score", "tenure", "estimated_salary"]].describe().T.round(2))

**Para el ETL:** conservar `credit_score`, `tenure` y `estimated_salary` como están. No hace falta añadir bandas de score, antigüedad o salario al modelo final.


### 2.3. ¿Qué columnas nuevas necesitamos?

Los cortes por score, salario y antigüedad ayudan a explorar, pero no dieron una división lo bastante clara como para guardarla en Power BI. Se conservan los campos originales.


## 3. Análisis por edad

Agrupo edades en 18–29, 30–39, 40–49, 50–59, 60–69 y 70+. También conservo la edad original para revisar casos puntuales.


In [ ]:
edad = df.groupby("age").agg(clientes=("customer_id","size"),bajas=("churn","sum")).reset_index()
edad["churn_pct"] = 100*edad.bajas/edad.clientes
display(edad.head(20))
bandas = pd.cut(df.age,[17,29,39,49,59,69,np.inf],labels=["18–29","30–39","40–49","50–59","60–69","70+"])
display(df.assign(edad_banda=bandas).groupby("edad_banda",observed=True).agg(clientes=("customer_id","size"),bajas=("churn","sum")))

### 3.1. Gráfico de edades

Comparo tasa de abandono y número de clientes. En el grupo de 70+ hay 151 personas; el resultado debe interpretarse con cuidado.


In [ ]:
edad_eda = resumen_grupos_exploratorio("edad_banda", bandas).rename(columns={"grupo_eda": "edad_banda"})
grafico_tasas_exploratorias(edad_eda, edad_eda.edad_banda.astype(str).tolist(), "Edad — churn por banda reproducible")

**Para el ETL:** crear `edad_banda` y mantener `age` sin cambios.


## 4. Análisis de saldo

Separo los saldos en cero y positivo. Los positivos se dividen en cuatro grupos usando sus cuartiles (25%, 50% y 75%). Los cortes se calculan a partir de los cuartiles de la muestra.


In [ ]:
positivo = df.loc[df.balance>0,"balance"]
print("Ceros:",int(df.balance.eq(0).sum())," Positivos:",len(positivo))
print("Cuartiles brutos:", positivo.quantile([.25,.5,.75]).to_dict())

### 4.1. Comparación por saldo

Reviso si separar el saldo cero y los cuartiles ayuda a ver diferencias de abandono.


In [ ]:
# Probar las bandas antes de llevarlas al ETL.
q_eda = positivo.quantile([.25, .5, .75], interpolation="linear")
q1_eda, q2_eda, q3_eda = (float(q_eda.loc[p]) for p in [.25, .5, .75])
fmt_eda = lambda v: f"{v:.2f}"
labels_eda = [
    f"Positivo Q1 (≤{fmt_eda(q1_eda)})", f"Positivo Q2 (≤{fmt_eda(q2_eda)})",
    f"Positivo Q3 (≤{fmt_eda(q3_eda)})", f"Positivo Q4 (>{fmt_eda(q3_eda)})"
]
saldo_eda = pd.Series("Cero", index=df.index, dtype="object")
mask_eda = df.balance.gt(0)
saldo_eda.loc[mask_eda] = pd.cut(
    df.loc[mask_eda, "balance"],
    bins=[0, q1_eda, q2_eda, q3_eda, np.inf], labels=labels_eda,
    include_lowest=True
).astype(str)
orden_saldo_eda = ["Cero"] + labels_eda
saldo_tabla_eda = resumen_grupos_exploratorio("saldo_banda", saldo_eda)
saldo_tabla_eda["grupo_eda"] = pd.Categorical(saldo_tabla_eda.grupo_eda, categories=orden_saldo_eda, ordered=True)
saldo_tabla_eda = saldo_tabla_eda.sort_values("grupo_eda").reset_index(drop=True)
display(saldo_tabla_eda)
grafico_tasas_exploratorias(saldo_tabla_eda, ["Cero", "Q1", "Q2", "Q3", "Q4"], "Saldo — cero y cuartiles positivos")
assert (saldo_tabla_eda.clientes.sum(), saldo_tabla_eda.bajas.sum()) == (10000, 2037)

**Para el ETL:** crear `saldo_estado` y `saldo_banda`. Mantener `balance` original. El saldo asociado a bajas no representa una pérdida financiera realizada.


### 4.2. Cruces entre variables

Una sola columna no siempre explica bien el perfil. Comparo actividad con productos, saldo con productos y actividad con saldo. También reviso tarjeta con productos para decidir si vale la pena agregar ese cruce.

Los cruces se calculan aquí desde el CSV; no modifico la tabla original.


In [ ]:
# Probar combinaciones entre columnas.
desc = df.assign(
    saldo_estado=np.where(df.balance.eq(0), "Cero", "Positivo"),
    vinculacion_tipo=np.where(df.products_number.eq(1), "Un producto", "Multiproducto"),
    actividad_etiqueta=df.active_member.map({0: "Inactivo", 1: "Activo"}),
    tarjeta_etiqueta=df.credit_card.map({0: "Sin tarjeta", 1: "Con tarjeta"})
)
desc["perfil_actividad_productos"] = desc.actividad_etiqueta + " / " + desc.vinculacion_tipo
desc["perfil_actividad_saldo"] = desc.actividad_etiqueta + " / " + desc.saldo_estado
desc["perfil_saldo_producto"] = desc.saldo_estado + " / " + desc.vinculacion_tipo
# Este cruce sólo se revisa aquí.
desc["perfil_tarjeta_productos"] = desc.tarjeta_etiqueta + " / " + desc.vinculacion_tipo

def desc_resumir(campos):
    out = (desc.groupby(campos, dropna=False).agg(
        clientes=("customer_id", "size"), bajas=("churn", "sum")
    ).reset_index())
    out["churn_pct"] = (100 * out.bajas / out.clientes).round(2)
    out["aporte_bajas_pct"] = (100 * out.bajas / df.churn.sum()).round(2)
    assert (out.clientes.sum(),out.bajas.sum()) == (len(df),int(df.churn.sum()))
    return out

for etiqueta, campo in [
    ("Actividad × productos", "perfil_actividad_productos"),
    ("Saldo × productos", "perfil_saldo_producto"),
    ("Actividad × saldo", "perfil_actividad_saldo"),
    ("Tarjeta × productos, candidata descartada", "perfil_tarjeta_productos")
]:
    print("\n" + etiqueta)
    display(desc_resumir([campo]))

# Comparar perfiles con saldo cero.
resumen_cero = desc_resumir(["saldo_estado", "vinculacion_tipo"])
resumen_cero = resumen_cero[resumen_cero.saldo_estado.eq("Cero")].copy()
display(resumen_cero)
assert int(resumen_cero.clientes.sum()) == int(df.balance.eq(0).sum())
grafico_tasas_exploratorias(
    resumen_cero, resumen_cero.vinculacion_tipo.tolist(),
    "Saldo cero — un producto frente a multiproducto"
)
# Revisar que la fuente no cambió.
assert df.shape == (10000, 12) and df.customer_id.is_unique

**Hallazgo:** entre clientes con saldo cero, los de un producto tienen 37.35% de abandono (338 de 905), mientras que los de varios productos tienen 5.97% (162 de 2,712). Por eso conviene conservar `perfil_saldo_producto`. El cruce de actividad y productos también ayuda a definir H1 y H3.

Con un solo producto, tener o no tarjeta da tasas parecidas (27.70% y 27.76%). Por eso conservo `credit_card` sin crear otra columna para ese cruce.

El grupo de cuatro productos requiere revisión porque es pequeño y presenta 100% de abandono en esta muestra.


### 4.3. Columnas que conservaremos

Durante la exploración se probaron varias columnas nuevas, pero **no todas hacen falta en el modelo final**. Estas diez son las que genera el ETL junto a las 12 originales:

| Columna nueva | Motivo |
|---|---|
| `saldo_estado` | Distinguir saldo cero y positivo |
| `edad_banda` | Comparar grupos de edad |
| `saldo_banda` | Comparar saldos por grupos |
| `vinculacion_tipo` | Separar un producto de varios |
| `perfil_actividad_saldo` | Cruzar actividad y saldo |
| `perfil_actividad_productos` | Cruzar actividad y productos |
| `perfil_saldo_producto` | Cruzar saldo y productos |
| `productos_alto_3o4` | Identificar 3–4 productos |
| `hipotesis` | Identificar H1, H2, H3 y Fuera |
| `simulable` | Marcar H1 y H3 como grupos del simulador |

Algunas son de apoyo y no tienen que aparecer en un gráfico.

Las alertas de calidad van en `CalidadCliente`; los nombres y códigos de país están en `DimPais`. No conservo ratios de saldo/salario o edad/antigüedad porque pueden llevar a conclusiones poco claras sin más información. Tampoco creo columnas repetidas cuando un cálculo de Power BI o un campo original es suficiente.


## 5. Grupos para las recomendaciones

Defino grupos que no se traslapan: **H1** (inactivo, un producto), **H2** (tres o cuatro productos), **H3** (activo, un producto) y **Fuera** (resto). Primero clasifico por las características del cliente y después calculo las bajas.


In [ ]:
h = np.select([(df.active_member==0)&(df.products_number==1),df.products_number>=3,(df.active_member==1)&(df.products_number==1)],["H1","H2","H3"],default="Fuera")
t = df.assign(hipotesis=h).groupby("hipotesis").agg(clientes=("customer_id","size"),bajas=("churn","sum"),saldo_observado=("balance","sum")).reset_index()
t["churn_pct"] = 100*t.bajas/t.clientes
t["aporte_bajas_pct"] = 100*t.bajas/df.churn.sum()
display(t)
assert dict(zip(t.hipotesis,t.clientes)) == {"Fuera":4590,"H1":2521,"H2":326,"H3":2563}

### 5.1. ¿Qué grupo conviene atender?

H1 tiene muchas bajas en número absoluto. H2 tiene una tasa muy alta, pero menos clientes. H3 puede ayudar a estudiar acciones preventivas. No basta con ordenar sólo por porcentaje.


In [ ]:
hip_eda = t.set_index("hipotesis").loc[["Fuera", "H1", "H2", "H3"]].reset_index()
grafico_tasas_exploratorias(hip_eda, hip_eda.hipotesis.tolist(), "Hipótesis — riesgo observado y tamaño")
resumen_hip_eda = hip_eda[["hipotesis", "clientes", "bajas", "churn_pct", "aporte_bajas_pct"]].copy()
resumen_hip_eda[["churn_pct", "aporte_bajas_pct"]] = resumen_hip_eda[["churn_pct", "aporte_bajas_pct"]].round(2)
display(resumen_hip_eda)
assert int(resumen_hip_eda.bajas.sum()) == int(df.churn.sum())

**Ideas a evaluar:** H1: piloto de reactivación. H2: revisar primero los datos y la experiencia de productos. H3: piloto preventivo. Estas acciones aún no tienen resultados reales; tendrían que medirse.


## 6. Radar de segmentos

Para profundizar el análisis, pruebo reglas con una o dos variables y al menos 100 clientes por grupo. Se obtienen 84 segmentos. Un cliente puede aparecer en varios, por lo que el total se calcula por cliente único.


In [ ]:
# Crear las reglas desde los datos.
# Revisar las mismas reglas del ETL.
base = df.copy()
base["saldo_estado"] = np.where(base.balance == 0, "Cero", "Positivo")
base["edad_banda"] = pd.cut(base.age,[17,29,39,49,59,69,np.inf],labels=["18–29","30–39","40–49","50–59","60–69","70+"]).astype(str)
base["vinculacion_tipo"] = np.where(base.products_number == 1, "Un producto", "Multiproducto")
base["perfil_actividad_saldo"] = base.active_member.map({0:"Inactivo",1:"Activo"}) + " / " + base.saldo_estado
base["perfil_actividad_productos"] = base.active_member.map({0:"Inactivo",1:"Activo"}) + " / " + base.vinculacion_tipo
base["perfil_saldo_producto"] = base.saldo_estado + " / " + base.vinculacion_tipo
radar_cols = ["country","active_member","products_number","credit_card","edad_banda","saldo_estado","vinculacion_tipo","perfil_actividad_saldo","perfil_actividad_productos","perfil_saldo_producto"]
cruces = [("country","active_member"),("country","products_number"),("active_member","products_number"),("edad_banda","active_member"),("edad_banda","products_number"),("saldo_estado","active_member"),("saldo_estado","products_number"),("credit_card","active_member")]
reglas = []; pertenencias = 0
for dims in [(x,) for x in radar_cols] + cruces:
    for _, grupo in base.groupby(list(dims),dropna=False):
        if len(grupo) < 100:
            continue
        reglas.append((dims,len(grupo),int(grupo.churn.sum())))
        pertenencias += len(grupo)
print("Radar generado por reglas:",len(reglas),"segmentos;",pertenencias,"pertenencias")
assert len(reglas) == 84 and pertenencias == 178915


### 6.1. Evitar contar dos veces a un cliente

Un cliente puede estar en el grupo Alemania y también en H1. Si sumara ambos grupos, podría contarlo dos veces. Por eso se necesita una tabla con la relación entre cada cliente y los segmentos que cumple.


In [ ]:
ids_alemania_eda = set(df.loc[df.country.eq("Germany"), "customer_id"])
ids_h1_eda = set(df.loc[(df.active_member.eq(0)) & (df.products_number.eq(1)), "customer_id"])
ids_union_eda = ids_alemania_eda | ids_h1_eda
ids_inter_eda = ids_alemania_eda & ids_h1_eda
bajas_union_eda = int(df.loc[df.customer_id.isin(ids_union_eda), "churn"].sum())
union_eda = pd.DataFrame([
    ("Clientes en Alemania", len(ids_alemania_eda)),
    ("Clientes en H1", len(ids_h1_eda)),
    ("Suma ingenua de pertenencias", len(ids_alemania_eda) + len(ids_h1_eda)),
    ("Clientes repetidos", len(ids_inter_eda)),
    ("Clientes únicos en la unión", len(ids_union_eda)),
    ("Bajas únicas en la unión", bajas_union_eda),
], columns=["concepto", "resultado"])
display(union_eda)
assert len(ids_union_eda) == len(ids_alemania_eda) + len(ids_h1_eda) - len(ids_inter_eda)
assert len(ids_union_eda) <= len(df)

**Para el ETL:** generar `RadarSegmentos` con las reglas y `RadarCliente.csv` con las pertenencias. Power BI calcula el total de clientes únicos al elegir varios grupos.


### 6.2. Qué usa el Radar

Los perfiles de actividad, saldo y productos ayudan a formar reglas que se pueden comparar por tamaño y tasa de abandono. El Radar es descriptivo: no predice quién abandonará.

El ETL guarda las reglas en `RadarSegmentos` y las relaciones cliente–segmento en `RadarCliente.csv`. Son archivos diferentes porque cumplen funciones distintas.


## 7. Conclusiones del análisis

El análisis permite priorizar por número de bajas, tasa y tamaño del grupo. Para H2 conviene revisar el origen del porcentaje tan alto. Para H1 y H3 se pueden plantear pruebas de retención.

El simulador de Power BI utiliza costos, presupuestos y mejoras **supuestas**; no representa ganancias ni efectos reales. El modelo final tiene ocho páginas y 92 medidas DAX.


### 7.1. Qué debe hacer el ETL

| Encontrado en la exploración | Preparación necesaria | Uso en Power BI |
|---|---|---|
| 10,000 clientes y 2,037 bajas | Revisar registros y mantener 12 columnas | Indicadores generales |
| País, actividad, productos, género y tarjeta | Conservar los campos originales | Comparaciones y filtros |
| Diferencias por edad | Crear `edad_banda` | Análisis por edad |
| Saldo cero y cuartiles positivos | Crear dos columnas de saldo | Segmentación por saldo |
| Score, salario y antigüedad | Mantener originales y crear alertas | Análisis y calidad |
| Perfiles H1, H2 y H3 | Etiquetar clientes | Priorización |
| Segmentos que se traslapan | Generar reglas y tabla puente | Radar sin duplicados |
| No hay fechas ni resultados de campañas | Separar los supuestos | Simulación económica |

El ETL toma estas decisiones y genera los datos para Power BI.


### 7.2. Archivos que utilizará Power BI

El segundo notebook produce un Excel de seis hojas y un CSV adicional:

| Hoja o archivo | Contenido |
|---|---|
| `FactClientes` | Clientes y columnas seleccionadas |
| `DimPais` | Nombres y códigos de país |
| `RadarSegmentos` | Reglas y resultados por segmento |
| `CalidadCliente` | Alertas de calidad |
| `Escenarios` | Supuestos del simulador |
| `HipotesisResumen` | Resumen de H1, H2, H3 y Fuera |
| `RadarCliente.csv` | Relación de clientes con segmentos |

Power BI también utiliza sus medidas DAX y parámetros internos. Los resultados de campañas no existen en la fuente.


In [ ]:
# Resumen de los archivos que prepara el ETL.
desc_contrato = pd.DataFrame([
    ("Modelo_Analitica_Clientes.xlsx", "FactClientes", 10000, "Clientes y métricas globales"),
    ("Modelo_Analitica_Clientes.xlsx", "DimPais", 3, "Normalización y filtros geográficos"),
    ("Modelo_Analitica_Clientes.xlsx", "RadarSegmentos", 84, "Reglas descriptivas"),
    ("Modelo_Analitica_Clientes.xlsx", "CalidadCliente", 10000, "Alertas QA por cliente"),
    ("Modelo_Analitica_Clientes.xlsx", "Escenarios", None, "Supuestos hipotéticos"),
    ("Modelo_Analitica_Clientes.xlsx", "HipotesisResumen", 4, "H1/H2/H3/Fuera"),
    ("RadarCliente.csv", "Pertenencias de Radar", 178915, "Puente de reglas y clientes")
], columns=["archivo", "dataset", "filas_esperadas", "uso_principal"])
display(desc_contrato)
assert desc_contrato.dataset.nunique() == 7
assert len(df) == 10000 and df.churn.sum() == 2037
assert len(reglas) == 84 and pertenencias == 178915

**Nota:** las cantidades verificadas corresponden al dataset proporcionado. Si se cambia la fuente, hay que revisar nuevamente los conteos y los límites de las bandas.


In [ ]:
print("Investigación descriptiva finalizada. La ejecución productiva se realiza desde 02_ETL_Parametrizado.ipynb.")
